# Free fall - analysis

A sphere falls under gravity only (no collision response: the world is stepped with
`integrateWithoutCollisions()`), for each solver and a log-spaced range of dt. The scene and the dt
range are defined in `main.cpp`. Its results:

- `results/benchmark.csv`: one row per (solver, dt), with the metrics below;
- `results/energy_drift.csv`: E(t) for a few dt values.

Run `./scripts/physics_benchmark.sh --scenarios Free_Fall` to produce them and execute this notebook.

In [ ]:
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, "..")
from analysis_utils import RESULTS, STYLE, expected_orders, export_pdf, plot_solvers, save, shade_from, time_series

df = pd.read_csv(RESULTS / "benchmark.csv")
ede = pd.read_csv(RESULTS / "energy_drift.csv")
print(f"benchmark.csv: {len(df)} rows, energy_drift.csv: {len(ede)} rows")

## Contact time error vs dt

**How it is made.** The contact time is the first step where the narrow phase detects the ground,
refined by linear interpolation of the height between the last two steps. The error is
`|t_numerical − t_analytical|`. Dotted lines: expected slope for each solver.

**What to expect.**
- Verlet and RK4 reproduce the parabola exactly at the steps (constant acceleration), so their
  error only comes from interpolating a parabola linearly: **slope 2**.
- Semi-implicit Euler positions are off by O(dt): **slope 1**.
- At the smallest dt, round-off accumulated over many steps can flatten the curves.

In [ ]:
data = df.dropna(subset=["contact_error"])
fig, ax = plt.subplots(figsize=(9, 5))
plot_solvers(ax, data, "dt", "contact_error")
expected_orders(ax, data, "dt", "contact_error", {"Euler": 1, "Verlet": 2, "RK4": 2})
ax.set_xlabel("dt  (s)")
ax.set_ylabel("|contact time error|  (s)")
ax.set_title("Contact time error vs dt")
ax.legend(ncol=2, fontsize=8)
save(fig, "plot_convergence")

## Energy drift vs dt

**How it is made.** Largest relative change of the mechanical energy (kinetic + potential) over the
steps before contact: `max |E(t) − E₀| / E₀`.

**What to expect.**
- Semi-implicit Euler places the sphere ½·g·dt² too low at every step, so it loses energy steadily:
  **slope 1**.
- Verlet and RK4 are exact for a constant acceleration: only round-off remains, near the double
  precision ε and growing slowly with the number of steps (∝ 1/dt).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
plot_solvers(ax, df, "dt", "max_energy_drift")
expected_orders(ax, df, "dt", "max_energy_drift", {"Euler": 1})
ax.axhline(np.finfo(float).eps, color="grey", ls="--", lw=1, label="double precision ε")
ax.set_xlabel("dt  (s)")
ax.set_ylabel("max |E(t) − E₀| / E₀  (before contact)")
ax.set_title("Energy drift vs dt")
ax.legend(ncol=2, fontsize=8)
save(fig, "plot_energy_drift_vs_dt")

## Cost vs precision

**How it is made.** Same error as the convergence plot, against the cost of the run: the number of
force evaluations, `ops_total` = steps × evaluations per step (1 for Euler and Verlet, 4 for RK4, as
counted in `main.cpp`).

**What to expect.** At a given cost (vertical line), the lowest curve is the most accurate solver
for that budget. A higher order pays off only if its error falls faster than its extra cost.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
plot_solvers(ax, df.dropna(subset=["contact_error"]), "ops_total", "contact_error")
ax.set_xlabel("Force evaluations over the run  (cost)")
ax.set_ylabel("|contact time error|  (s)")
ax.set_title("Cost vs precision")
ax.legend(fontsize=8)
save(fig, "plot_cost_vs_precision")

## CPU time vs dt

**How it is made.** Wall time of the simulation loop, in µs, for each run. Runs shorter than
the timer resolution (1 µs) read 0 and are not shown. The dotted lines have slope −1.

**What to expect.** The time is proportional to the number of steps, so to 1/dt (slope −1). RK4
evaluates the forces 4 times per step, but the rest of the per-step work does not scale with it, so
RK4 should cost less than 4× Euler.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
plot_solvers(ax, df, "dt", "cpu_us")
expected_orders(ax, df, "dt", "cpu_us", {"Euler": -1, "Verlet": -1, "RK4": -1})
ax.set_xlabel("dt  (s)")
ax.set_ylabel("CPU time  (µs)")
ax.set_title("CPU time vs dt")
ax.legend(ncol=2, fontsize=8)
save(fig, "plot_cpu_time")

## Energy over time

**How it is made.** `|E(t) − E₀| / E₀` along the run, sub-sampled, for the dt values recorded by
`main.cpp`, until contact.

**What to expect.** Euler grows steadily from the first step. Verlet and RK4 stay at round-off
level. On this log scale of an absolute value, sharp dips are points where `E(t) − E₀` changes sign.

In [ ]:
fig = time_series(ede)
fig.suptitle("Energy drift over time", y=1.0)
save(fig, "plot_energy_time_series")

## PDF report

In [ ]:
export_pdf()